# NLP - Text Classification using:

- CountVectorizer
- NaiveBiase
- Un-Processed and Processed Data

### Text Classification Pipeline Steps:
- Data Acquisition
- Text Pre Processing
- Text Vectorization
- Modeling (Naive Bayes, Random Forrest, Logistic Regression)
- Model Evaluation (Accuracy Score, Confusion Matrix)
- Deploy Model

In [79]:
from sklearn.feature_extraction.text import CountVectorizer
import spacy

In [80]:
nlp = spacy.load('en_core_web_sm')

# Corpus of Text

In [81]:
# Generate a long corpus of text data
corpus = [
    """
    The train arrived five minutes earlier than expected, which unsettled Martin more than it should have. He preferred systems that behaved predictably. The platform was quiet except for a flickering fluorescent light and the sound of luggage wheels dragging across uneven concrete. He checked his phone, not because he had a message, but because the gesture gave him something to anchor to. A woman in a gray coat stepped off the third carriage. She scanned the crowd as though searching for someone who had once mattered deeply. Their eyes met for a brief second. Recognition? Unlikely. Projection? Possibly. Memory is unreliable under fluorescent lighting. By the time Martin reached the exit, he had convinced himself that coincidence is merely a narrative convenience humans use to avoid confronting randomness.
    """
]

# Method to Vectorize the text

In [82]:
def vectorize(text):
    # Create a Bag-of-Words model with unigrams, bigrams, and trigrams
    vect = CountVectorizer(ngram_range=(1,3)) #, stop_words='english') #, max_features=2**16)
    vectorized_text = vect.fit_transform([text])
    print(f"Vocabulary size: {len(vect.vocabulary_)}")
    print(type(vect.vocabulary_))
    
    first_5_items = [(k, vect.vocabulary_[k]) for i, k in enumerate(vect.vocabulary_) if i < 5]
    last_5_items = [(k, vect.vocabulary_[k]) for i, k in enumerate(vect.vocabulary_) if i >= len(vect.vocabulary_) - 5]
    print(f"First 5 vocabulary items:\n{first_5_items}")
    print(f"Last 5 vocabulary items:\n{last_5_items}")

    return vectorized_text

## Vectorize the unprocessed text

In [83]:
print(f"Length of the first document in the corpus: {len(corpus[0])}")
vectorized_text = vectorize(corpus[0])
print(f"Shape of the vectorized first document: {vectorized_text.shape}")

Length of the first document in the corpus: 823
Vocabulary size: 340
<class 'dict'>
First 5 vocabulary items:
[('the', 268), ('train', 304), ('arrived', 9), ('five', 82), ('minutes', 179)]
Last 5 vocabulary items:
[('convenience humans use', 54), ('humans use to', 138), ('use to avoid', 324), ('to avoid confronting', 301), ('avoid confronting randomness', 17)]
Shape of the vectorized first document: (1, 340)


## Pre Processing function to remove stop words, punctuation and lemmatize.

In [84]:
def preprocess(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return (' '.join(tokens)).strip()

### Pre Process the text to remove noise

In [85]:
processed_text = [preprocess(text) for text in corpus]

print(len(processed_text[0]))
print(processed_text)

479
['train arrive minute early expect unsettle Martin prefer system behave predictably platform quiet flickering fluorescent light sound luggage wheel drag uneven concrete check phone message gesture give anchor woman gray coat step carriage scan crowd search matter deeply eye meet brief second recognition unlikely Projection possibly memory unreliable fluorescent lighting time Martin reach exit convince coincidence merely narrative convenience human use avoid confront randomness']


### Vectorize the denoised text

In [86]:
vectorized_text_text = [vectorize(text) for text in processed_text]
print(f"Shape of the vectorized first document in the list: {vectorized_text_text[0].shape}")

Vocabulary size: 187
<class 'dict'>
First 5 vocabulary items:
[('train', 163), ('arrive', 3), ('minute', 108), ('early', 47), ('expect', 53)]
Last 5 vocabulary items:
[('narrative convenience human', 113), ('convenience human use', 34), ('human use avoid', 78), ('use avoid confront', 180), ('avoid confront randomness', 8)]
Shape of the vectorized first document in the list: (1, 187)


# News Category Classification

In [87]:
import pandas as pd
import numpy as np

In [88]:
df_news = pd.read_json('../data/news_dataset.json')
print(df_news.head())

                                                text  category
0  Watching Schrödinger's Cat Die University of C...   SCIENCE
1     WATCH: Freaky Vortex Opens Up In Flooded Lake    SCIENCE
2  Entrepreneurs Today Don't Need a Big Budget to...  BUSINESS
3  These Roads Could Recharge Your Electric Car A...  BUSINESS
4  Civilian 'Guard' Fires Gun While 'Protecting' ...     CRIME


In [89]:
df_news.info()

<class 'pandas.DataFrame'>
Index: 12695 entries, 0 to 12694
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   text      12695 non-null  str  
 1   category  12695 non-null  str  
dtypes: str(2)
memory usage: 297.5 KB


In [90]:
df_news['category'].value_counts()

category
BUSINESS    4254
SPORTS      4167
CRIME       2893
SCIENCE     1381
Name: count, dtype: int64

# Undersample the other classes to math the min class count

In [91]:
min_samples = df_news['category'].value_counts().sort_values()

#Take the first items in the sorted value
min_samples_class = min_samples.index[0]
min_samples = min_samples[min_samples > 0].values[0]

print(f"Category with the minimum number of samples: {min_samples_class} = {min_samples}")

Category with the minimum number of samples: SCIENCE = 1381


In [92]:
seed = 42
df_business = df_news[df_news['category'] == "BUSINESS"].sample(min_samples, random_state=seed)
df_sports = df_news[df_news['category'] == "SPORTS"].sample(min_samples, random_state=seed)
df_crime = df_news[df_news['category'] == "CRIME"].sample(min_samples, random_state=seed)
df_science = df_news[df_news['category'] == "SCIENCE"].sample(min_samples, random_state=seed)

# Our random undersampling is done

In [93]:
df_balanced = pd.concat([df_business, df_sports, df_crime, df_science], axis=0).reset_index(drop=True)
df_balanced['category'].value_counts()

category
BUSINESS    1381
SPORTS      1381
CRIME       1381
SCIENCE     1381
Name: count, dtype: int64

# Feature engineer a new target column

In [94]:
df_balanced['Category_num'] = df_balanced['category'].map({
    "BUSINESS" : 0,
    "SPORTS" : 1,
    "CRIME" : 2,
    "SCIENCE" : 3
})

df_balanced.info(), df_balanced.sample(frac=1).head()

<class 'pandas.DataFrame'>
RangeIndex: 5524 entries, 0 to 5523
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   text          5524 non-null   str  
 1   category      5524 non-null   str  
 2   Category_num  5524 non-null   int64
dtypes: int64(1), str(2)
memory usage: 129.6 KB


(None,
                                                    text  category  \
 1147  How Marketing Leaders Can Secure a Seat in the...  BUSINESS   
 1833  Tom Brady Cuts Off Interview After Host Calls ...    SPORTS   
 4035  Attackers Sentenced To 3 Years In Prison For H...     CRIME   
 1964  Anthony Davis 'Guarantees' Kentucky's National...    SPORTS   
 443   Uber Admits Mistake In Accepting Sex Assault S...  BUSINESS   
 
       Category_num  
 1147             0  
 1833             1  
 4035             2  
 1964             1  
 443              0  )

# Fit a model

In [95]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

### Train Test Split

In [96]:
X_train, X_test, y_train, y_test = train_test_split(
    df_balanced['text'],
    df_balanced['Category_num'],
    test_size=0.2,
    random_state=seed,
    stratify=df_balanced['Category_num']
)

print(y_test.value_counts())
X_train.shape, X_test.shape, y_train.shape, y_test.shape 

Category_num
2    277
0    276
1    276
3    276
Name: count, dtype: int64


((4419,), (1105,), (4419,), (1105,))

### Fit a Naivebiase using a uni-gram vectorizer (Unprocessed Data)

In [97]:
pipe_line = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('classifier', MultinomialNB())
]).fit(X_train, y_train)
y_pred = pipe_line.predict(X_test)
print(classification_report(y_test, y_pred, target_names=df_balanced['category'].unique()))

              precision    recall  f1-score   support

    BUSINESS       0.78      0.92      0.84       276
      SPORTS       0.92      0.85      0.88       276
       CRIME       0.91      0.89      0.90       277
     SCIENCE       0.89      0.82      0.85       276

    accuracy                           0.87      1105
   macro avg       0.88      0.87      0.87      1105
weighted avg       0.88      0.87      0.87      1105



### Fit a Naivebiase using a bi-gram vectorizer (Unprocessed Data)

In [98]:
pipe_line = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1,2))),
    ('classifier', MultinomialNB())
]).fit(X_train, y_train)
y_pred = pipe_line.predict(X_test)
print(classification_report(y_test, y_pred, target_names=df_balanced['category'].unique()))

              precision    recall  f1-score   support

    BUSINESS       0.73      0.95      0.82       276
      SPORTS       0.92      0.83      0.87       276
       CRIME       0.91      0.88      0.89       277
     SCIENCE       0.93      0.79      0.85       276

    accuracy                           0.86      1105
   macro avg       0.87      0.86      0.86      1105
weighted avg       0.87      0.86      0.86      1105



# Pre process the text and then fit and predict

In [99]:
df_balanced['processed_text'] = df_balanced['text'].apply(preprocess)
print(df_balanced[['text', 'processed_text']].sample(5))

                                                   text  \
125   5 Tips for a Mindful 2015 The start of another...   
251   Why Elon Musk’s Plan To Merge Tesla With Solar...   
1318  Mulligan Time: 5 Foolproof Ways to Tap a New C...   
4709  We Now Know How Tiny Lizards Defy Gravity With...   
2270  Pistons Coach Talks Dealing With Burnout In Th...   

                                         processed_text  
125   5 tip mindful 2015 start year great time check...  
251   Elon Musk plan merge Tesla SolarCity probably ...  
1318  Mulligan Time 5 Foolproof Ways tap New Career ...  
4709  know Tiny Lizards Defy Gravity Gargantuan Tong...  
2270  piston Coach Talks deal Burnout NBA say game t...  


In [100]:
X_train, X_test, y_train, y_test = train_test_split(
    df_balanced['processed_text'],
    df_balanced['Category_num'],
    test_size=0.2,
    random_state=seed,
    stratify=df_balanced['Category_num']
)

print(y_test.value_counts())
X_train.shape, X_test.shape, y_train.shape, y_test.shape 

Category_num
2    277
0    276
1    276
3    276
Name: count, dtype: int64


((4419,), (1105,), (4419,), (1105,))

### Fit using uni-gram vectorizer (Pre-processed Data)

In [101]:
pipe_line = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('classifier', MultinomialNB())
]).fit(X_train, y_train)
y_pred = pipe_line.predict(X_test)
print(classification_report(y_test, y_pred, target_names=df_balanced['category'].unique()))

              precision    recall  f1-score   support

    BUSINESS       0.85      0.90      0.87       276
      SPORTS       0.91      0.88      0.89       276
       CRIME       0.88      0.94      0.91       277
     SCIENCE       0.92      0.85      0.88       276

    accuracy                           0.89      1105
   macro avg       0.89      0.89      0.89      1105
weighted avg       0.89      0.89      0.89      1105



### Fit using Bi-gram vectorizer (Pre-processed Data)

In [102]:
pipe_line = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1, 2))),
    ('classifier', MultinomialNB())
]).fit(X_train, y_train)
y_pred = pipe_line.predict(X_test)
print(classification_report(y_test, y_pred, target_names=df_balanced['category'].unique()))

              precision    recall  f1-score   support

    BUSINESS       0.83      0.91      0.87       276
      SPORTS       0.92      0.87      0.89       276
       CRIME       0.88      0.94      0.91       277
     SCIENCE       0.93      0.82      0.87       276

    accuracy                           0.89      1105
   macro avg       0.89      0.89      0.89      1105
weighted avg       0.89      0.89      0.89      1105



### Fit a hex-gram vectorizer (Pre-processed Data)

In [103]:
pipe_line = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1, 6))),
    ('classifier', MultinomialNB())
]).fit(X_train, y_train)
y_pred = pipe_line.predict(X_test)
print(classification_report(y_test, y_pred, target_names=df_balanced['category'].unique()))

              precision    recall  f1-score   support

    BUSINESS       0.83      0.92      0.87       276
      SPORTS       0.92      0.87      0.89       276
       CRIME       0.88      0.94      0.91       277
     SCIENCE       0.93      0.82      0.87       276

    accuracy                           0.89      1105
   macro avg       0.89      0.89      0.89      1105
weighted avg       0.89      0.89      0.89      1105



# Observation

Using Preprocessed data we observed a significant improvements. However, there was no change in the performance when validated using Unigram, Bigram or even higher dimensions. 

Hence using a Unigram Vectorizer with NaiveBiase over processed data is recommended.